# Grounded-Physics LM Adapter Training

**Copyright (c) 2026 Anonymous. All rights reserved.**

**Author:** Anonymous

**PROPRIETARY AND CONFIDENTIAL.** This software is provided for academic review and research purposes only. Unauthorized copying, modification, distribution, or use of this software, via any medium, is strictly prohibited without prior written permission from Anonymous.

---

This notebook trains the Grounded-Physics LM Adapter on top of a pre-trained PhysicsFormer model.

## Architecture
- **PhysicsFormer** (frozen): Encodes physics states into embeddings
- **Adapter MLP**: Projects physics embeddings to LLM prefix tokens
- **DistilGPT-2** (82M): Generates answers from prefix + question
- **Specialized Heads**: Numerical (6 outputs) + Descriptive (multi-head classification)

## Training Phases (Progressive Unfreezing)
1. **Phase 1**: Train adapter + heads (LLM frozen)
2. **Phase 2**: Unfreeze LLM output layer
3. **Phase 3**: Unfreeze full LLM

## Key Features (Physics Former V2 Technologies)
- **AMP (Automatic Mixed Precision)** - 2x faster on A100
- **OneCycleLR Scheduler** - Better convergence
- **PlateauTracker + Catapult** - Escape training plateaus
- **Contrastive Loss** - Prevent modality collapse (model ignoring physics)
- **Physics Usage Validation** - Detect if model actually uses physics
- **Plateau epochs don't count against progression** - Fair chance for catapult

## Prerequisites
- Trained PhysicsFormer checkpoint (from `train_physics_former_v2.ipynb`)
- Physics HDF5 data files

## Hardware Optimization
- **A100 80GB**: Batch size 64, 4 workers, pin memory enabled

In [ ]:
# ============================================================
# CELL 1: CONFIGURATION (MATCHING colab_train_adapter.ipynb)
# ============================================================
# Uses HDF5 data format and physics_former_best.pt (state_dim=35, hidden_dim=768)

# GDrive base path for adapter data
GDRIVE_ADAPTER_DATA = "/content/drive/MyDrive/physics_llm"

# Data paths (HDF5 format - from working notebook)
DATA_PATH = f'{GDRIVE_ADAPTER_DATA}/data'
CHECKPOINT_PATH = f'{GDRIVE_ADAPTER_DATA}/checkpoints'
HDF5_DATA_PATH = f'{DATA_PATH}/clevrer_training_expanded.h5'

# Physics checkpoint - MUST use physics_former_best.pt (state_dim=35, hidden_dim=768)
PHYSICS_CHECKPOINT = f'{CHECKPOINT_PATH}/physics_former_best.pt'

# Output directory for adapter checkpoints
OUTPUT_DIR = CHECKPOINT_PATH

# ============================================================
# TRAINING CONFIG (matching colab_train_adapter.ipynb)
# ============================================================
BATCH_SIZE = 4  # Small batch size for stable training
GRADIENT_ACCUMULATION = 1
NUM_WORKERS = 0  # Set to 0 to avoid multiprocessing errors in Colab
PIN_MEMORY = True

NUM_PREFIX_TOKENS = 64

# Physics model dimensions - auto-detected from checkpoint
# These are just defaults, actual values come from checkpoint
STATE_DIM = 35  # Will be detected from checkpoint
EMBED_DIM = 768  # Will be detected from checkpoint (hidden_dim)
NUM_HEADS = 24  # Will be detected from checkpoint
NUM_LAYERS = 8  # Will be detected from checkpoint
FF_DIM = 768 * 4  # hidden_dim * 4
MAX_OBJECTS = 20

# Phase-specific learning rates (matching colab_train_adapter.ipynb)
PHASE1_LR = 1e-4
PHASE2_LR = 5e-5
PHASE3_LR = 1e-5

# Contrastive loss - prevents physics collapse
USE_CONTRASTIVE = True
CONTRASTIVE_WEIGHT = 0.1

# Physics validation - monitors physics usage
VALIDATE_PHYSICS_EVERY = 1  # Every epoch
PHYSICS_SIM_WARNING_THRESHOLD = 0.9  # Warn if similarity > 0.9

# Question types to focus on (causal reasoning - skip descriptive)
FOCUS_QUESTION_TYPES = ['counterfactual', 'causal', 'future', 'prediction']
MAX_TRAIN_SAMPLES = 20000  # Match colab_train_adapter

# Mixed precision
USE_AMP = True

print("="*70)
print("CONFIGURATION (matching colab_train_adapter.ipynb)")
print("="*70)
print(f"GDrive data path: {GDRIVE_ADAPTER_DATA}")
print(f"HDF5 Data: {HDF5_DATA_PATH}")
print(f"Physics checkpoint: {PHYSICS_CHECKPOINT}")
print(f"Output directory: {OUTPUT_DIR}")
print()
print(f"Batch size: {BATCH_SIZE}")
print(f"Prefix tokens: {NUM_PREFIX_TOKENS}")
print(f"Contrastive loss: {USE_CONTRASTIVE} (weight={CONTRASTIVE_WEIGHT})")
print(f"Physics validation: every {VALIDATE_PHYSICS_EVERY} epochs")
print("="*70)

In [2]:
# ============================================================
# CELL 2: IMPORTS AND SETUP
# ============================================================
# Fully self-contained - no companion bundle needed.

import os
import sys
import json
import subprocess
import math
import re
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from enum import Enum

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import numpy as np
import h5py

# Mount Google Drive if on Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_COLAB = True
    print("Running on Google Colab")
except:
    ON_COLAB = False
    print("Running locally")

# Install required external deps
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers", "accelerate", "tqdm", "h5py"
])

from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Mounted at /content/drive
Running on Google Colab
PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.2 GB


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


In [5]:
# ============================================================
# CELL 5: ADAPTER ENUMS AND VOCABULARIES
# ============================================================

class OutputType(Enum):
    CATEGORICAL = "categorical"
    NUMERICAL = "numerical"
    DESCRIPTIVE = "descriptive"

# CLEVRER descriptive answer vocabulary (21 classes)
CLEVRER_DESCRIPTIVE_VOCAB = [
    "0", "1", "2", "3", "4", "5",  # Numbers (count)
    "yes", "no",  # Yes/No (exist)
    "brown", "red", "purple", "cyan", "gray", "green", "blue", "yellow",  # Colors
    "cylinder", "sphere", "cube",  # Shapes
    "metal", "rubber"  # Materials
]
CLEVRER_ANSWER_TO_IDX = {ans: idx for idx, ans in enumerate(CLEVRER_DESCRIPTIVE_VOCAB)}
CLEVRER_IDX_TO_ANSWER = {idx: ans for idx, ans in enumerate(CLEVRER_DESCRIPTIVE_VOCAB)}

class DescriptiveSubtype(Enum):
    COUNT = "count"
    EXIST = "exist"
    QUERY_COLOR = "color"
    QUERY_SHAPE = "shape"
    QUERY_MATERIAL = "material"

SUBTYPE_VOCABS = {
    DescriptiveSubtype.COUNT: ["0", "1", "2", "3", "4", "5"],
    DescriptiveSubtype.EXIST: ["yes", "no"],
    DescriptiveSubtype.QUERY_COLOR: ["brown", "red", "purple", "cyan", "gray", "green", "blue", "yellow"],
    DescriptiveSubtype.QUERY_SHAPE: ["cylinder", "sphere", "cube"],
    DescriptiveSubtype.QUERY_MATERIAL: ["metal", "rubber"],
}

def classify_descriptive_subtype(question_text: str) -> DescriptiveSubtype:
    q_lower = question_text.lower().strip()
    if "how many" in q_lower:
        return DescriptiveSubtype.COUNT
    elif "what color" in q_lower or "what is the color" in q_lower:
        return DescriptiveSubtype.QUERY_COLOR
    elif "what shape" in q_lower or "what is the shape" in q_lower:
        return DescriptiveSubtype.QUERY_SHAPE
    elif "what material" in q_lower or "what is the material" in q_lower:
        return DescriptiveSubtype.QUERY_MATERIAL
    elif any(p in q_lower for p in ["is there", "are there", "any"]):
        return DescriptiveSubtype.EXIST
    return DescriptiveSubtype.EXIST

class CLEVRERQuestionCategory(Enum):
    DESCRIPTIVE = "descriptive"
    EXPLANATORY = "explanatory"
    PREDICTIVE = "predictive"
    COUNTERFACTUAL = "counterfactual"

DESCRIPTIVE_PATTERNS = ["how many", "what color", "what shape", "what material",
    "what is the color", "what is the shape", "what is the material",
    "are there any", "is there a", "are there", "is there"]
EXPLANATORY_PATTERNS = ["what caused", "responsible for", "why did", "what made",
    "which of the following is responsible"]
PREDICTIVE_PATTERNS = ["what will happen", "which event will happen", "will the", "what happens next"]
COUNTERFACTUAL_PATTERNS = ["what if", "without the", "if the .* is removed",
    "if the .* were removed", "if we remove"]

def classify_clevrer_question(question_text: str) -> CLEVRERQuestionCategory:
    q_lower = question_text.lower().strip()
    for pattern in COUNTERFACTUAL_PATTERNS:
        if re.search(pattern, q_lower):
            return CLEVRERQuestionCategory.COUNTERFACTUAL
    for pattern in EXPLANATORY_PATTERNS:
        if pattern in q_lower:
            return CLEVRERQuestionCategory.EXPLANATORY
    for pattern in PREDICTIVE_PATTERNS:
        if pattern in q_lower:
            return CLEVRERQuestionCategory.PREDICTIVE
    for pattern in DESCRIPTIVE_PATTERNS:
        if pattern in q_lower:
            return CLEVRERQuestionCategory.DESCRIPTIVE
    return CLEVRERQuestionCategory.DESCRIPTIVE

print("Adapter enums and vocabularies defined")

Adapter enums and vocabularies defined


In [6]:
# ============================================================
# CELL 6: ADAPTER HEADS (Descriptive + Numerical)
# ============================================================

class DescriptiveSubHead(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int):
        super().__init__()
        self.num_classes = num_classes
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_classes)
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.1)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

class DescriptiveHead(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.shared_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.count_head = DescriptiveSubHead(hidden_dim, hidden_dim // 2, num_classes=6)
        self.exist_head = DescriptiveSubHead(hidden_dim, hidden_dim // 2, num_classes=2)
        self.color_head = DescriptiveSubHead(hidden_dim, hidden_dim // 2, num_classes=8)
        self.shape_head = DescriptiveSubHead(hidden_dim, hidden_dim // 2, num_classes=3)
        self.material_head = DescriptiveSubHead(hidden_dim, hidden_dim // 2, num_classes=2)
        self.subtype_vocabs = SUBTYPE_VOCABS
        for m in self.shared_encoder.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.1)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _get_head_for_subtype(self, subtype):
        return {DescriptiveSubtype.COUNT: self.count_head, DescriptiveSubtype.EXIST: self.exist_head,
                DescriptiveSubtype.QUERY_COLOR: self.color_head, DescriptiveSubtype.QUERY_SHAPE: self.shape_head,
                DescriptiveSubtype.QUERY_MATERIAL: self.material_head}.get(subtype, self.exist_head)

    def forward(self, physics_features, subtype=None):
        shared = self.shared_encoder(physics_features)
        if subtype is None:
            return shared
        return self._get_head_for_subtype(subtype)(shared)

    def predict_batch(self, physics_features, question_texts):
        subtypes = [classify_descriptive_subtype(q) for q in question_texts]
        if len(set(subtypes)) == 1:
            subtype = subtypes[0]
            logits = self.forward(physics_features, subtype)
            pred_indices = torch.argmax(logits, dim=-1).tolist()
            vocab = self.subtype_vocabs[subtype]
            return [vocab[idx] for idx in pred_indices]
        answers = []
        for i, (subtype, q) in enumerate(zip(subtypes, question_texts)):
            logits = self.forward(physics_features[i:i+1], subtype)
            pred_idx = torch.argmax(logits, dim=-1).item()
            answers.append(self.subtype_vocabs[subtype][pred_idx])
        return answers

    def compute_loss(self, physics_features, question_texts, answer_texts, label_smoothing=0.1):
        total_loss = torch.tensor(0.0, device=physics_features.device)
        count = 0
        for i, (q, a) in enumerate(zip(question_texts, answer_texts)):
            subtype = classify_descriptive_subtype(q)
            logits = self.forward(physics_features[i:i+1], subtype)
            vocab = self.subtype_vocabs[subtype]
            a_lower = a.lower().strip()
            if a_lower in vocab:
                target = torch.tensor([vocab.index(a_lower)], device=logits.device)
                total_loss = total_loss + F.cross_entropy(logits, target, label_smoothing=label_smoothing)
                count += 1
        return total_loss / max(count, 1)

class NumericalHead(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 256, num_outputs: int = 6):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_outputs)
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.1)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, physics_features):
        return self.network(physics_features)

print("DescriptiveHead and NumericalHead defined")

DescriptiveHead and NumericalHead defined


In [ ]:
# ============================================================
# CELL 7: GROUNDED PHYSICS LM (Main Class)
# ============================================================

class GroundedPhysicsLM(nn.Module):
    """
    Grounded Physics Language Model

    Combines physics understanding with language generation:
    - PhysicsFormer encodes physics states into embeddings
    - Adapter projects physics embeddings to LLM prefix tokens
    - DistilGPT-2 generates answers from prefix + question
    - Specialized heads for numerical and descriptive outputs
    """

    LLM_CONFIGS = {
        "distilgpt2": {"dim": 768, "params": "82M"},
        "gpt2": {"dim": 768, "params": "124M"},
    }

    def __init__(self, physics_model, physics_dim: int = 768, llm_name: str = "distilgpt2",
                 num_prefix_tokens: int = 64, freeze_physics: bool = True, freeze_llm: bool = True,
                 input_noise: float = 0.01):
        super().__init__()
        if llm_name not in self.LLM_CONFIGS:
            raise ValueError(f"Unknown LLM: {llm_name}")

        self.llm_name = llm_name
        self.llm_dim = self.LLM_CONFIGS[llm_name]["dim"]
        self.physics_dim = physics_dim
        self.num_prefix_tokens = num_prefix_tokens
        self.input_noise = input_noise

        self.physics_model = physics_model
        if freeze_physics:
            self.physics_model.eval()
            for param in self.physics_model.parameters():
                param.requires_grad = False

        self.adapter = nn.Sequential(
            nn.Linear(physics_dim, self.llm_dim),
            nn.LayerNorm(self.llm_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(self.llm_dim, self.llm_dim * num_prefix_tokens),
            nn.LayerNorm(self.llm_dim * num_prefix_tokens),
            nn.Tanh()
        )
        for m in self.adapter.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.01)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

        self.numerical_head = NumericalHead(input_dim=physics_dim, hidden_dim=768, num_outputs=6)
        self.descriptive_head = DescriptiveHead(input_dim=physics_dim, hidden_dim=512)

        print(f"Loading LLM: {self.llm_name}...")
        self.llm = AutoModelForCausalLM.from_pretrained(self.llm_name)
        self.tokenizer = AutoTokenizer.from_pretrained(self.llm_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        if freeze_llm:
            self.llm.eval()
            for param in self.llm.parameters():
                param.requires_grad = False

        print(f"GroundedPhysicsLM: LLM={llm_name}, physics_dim={physics_dim}, prefix_tokens={num_prefix_tokens}")

    def extract_physics_features(self, physics_states, object_mask):
        """Extract features from physics model.
        
        NOTE: Velocity augmentation is handled INSIDE the physics model (encode_physics).
        Do NOT augment here to avoid double augmentation.
        
        Args:
            physics_states: [B, T, N, state_dim] or [B, N, state_dim] - raw physics states
            object_mask: [B, N] or [B, T, N] - object validity mask
            
        Returns:
            features: [B, physics_dim] - pooled physics embeddings
        """
        if self.training and self.input_noise > 0:
            physics_states = physics_states + torch.randn_like(physics_states) * self.input_noise

        with torch.no_grad() if not self.physics_model.training else torch.enable_grad():
            # Pass raw states to encoder (model handles velocity augmentation internally)
            obj_emb = self.physics_model.encode_physics(physics_states, object_mask)

            # obj_emb: [batch, num_objects, embed_dim] or [batch, seq_len, num_objects, embed_dim]
            if obj_emb.dim() == 4:
                if object_mask is not None:
                    mask_exp = object_mask.unsqueeze(1).unsqueeze(-1) if object_mask.dim() == 2 else object_mask.unsqueeze(-1)
                    obj_emb = obj_emb * mask_exp
                    features = obj_emb.sum(dim=2) / object_mask.sum(dim=-1, keepdim=True).clamp(min=1).unsqueeze(-1)
                    features = features.mean(dim=1)
                else:
                    features = obj_emb.mean(dim=(1, 2))
            elif obj_emb.dim() == 3:
                num_emb_objects = obj_emb.shape[1]
                if object_mask is not None:
                    num_mask_objects = object_mask.shape[-1]
                    if num_emb_objects != num_mask_objects:
                        if num_emb_objects > num_mask_objects:
                            obj_emb = obj_emb[:, :num_mask_objects, :]
                        else:
                            pad = torch.zeros(obj_emb.shape[0], num_mask_objects - num_emb_objects, obj_emb.shape[2], device=obj_emb.device)
                            obj_emb = torch.cat([obj_emb, pad], dim=1)
                    mask_exp = object_mask.unsqueeze(-1) if object_mask.dim() == 2 else object_mask[:, 0, :].unsqueeze(-1)
                    features = (obj_emb * mask_exp).sum(dim=1) / object_mask.sum(dim=-1, keepdim=True).clamp(min=1)
                else:
                    features = obj_emb.mean(dim=1)
            else:
                features = obj_emb
            return features


    def create_prefix_tokens(self, physics_features):
        batch_size = physics_features.size(0)
        return self.adapter(physics_features).view(batch_size, self.num_prefix_tokens, self.llm_dim)

    def predict_numerical(self, physics_states, object_mask):
        features = self.extract_physics_features(physics_states, object_mask)
        outputs = self.numerical_head(features)
        return {"distance": outputs[:, 0], "speed": outputs[:, 1], "time_to_collision": outputs[:, 2],
                "kinetic_energy": outputs[:, 3], "momentum": outputs[:, 4], "object_count": outputs[:, 5]}

    def _compute_question_lengths(self, question_text):
        return [len(self.tokenizer.encode(q + " Answer:", add_special_tokens=False)) for q in question_text]

    def forward(self, physics_states, object_mask, question_text, max_length=50):
        batch_size = physics_states.size(0)
        device = physics_states.device

        features = self.extract_physics_features(physics_states, object_mask)
        prefix = self.create_prefix_tokens(features)

        prompted = [q + " Answer:" for q in question_text]
        tokens = self.tokenizer(prompted, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

        q_embeds = self.llm.transformer.wte(tokens.input_ids)
        combined = torch.cat([prefix, q_embeds], dim=1)
        mask = torch.cat([torch.ones(batch_size, self.num_prefix_tokens, device=device), tokens.attention_mask], dim=1)

        outputs = self.llm.generate(inputs_embeds=combined, attention_mask=mask, max_new_tokens=max_length,
                                    temperature=0.7, top_p=0.9, do_sample=True, pad_token_id=self.tokenizer.eos_token_id)

        answers = []
        for text in self.tokenizer.batch_decode(outputs, skip_special_tokens=True):
            answers.append(text.split("Answer:")[-1].strip() if "Answer:" in text else text.strip())
        return answers

print("GroundedPhysicsLM defined (with GNS-style velocity augmentation)")

In [8]:
# ============================================================
# CELL 8: GROUNDED PHYSICS LM - LOSS AND TRAINING METHODS
# ============================================================

# Add methods to GroundedPhysicsLM
def _adapter_compute_loss(self, physics_states, object_mask, question_text, answer_text):
    batch_size = physics_states.size(0)
    device = physics_states.device
    features = self.extract_physics_features(physics_states, object_mask)
    prefix = self.create_prefix_tokens(features)

    full_text = [q + " Answer: " + a for q, a in zip(question_text, answer_text)]
    tokens = self.tokenizer(full_text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    q_lengths = self._compute_question_lengths(question_text)

    text_embeds = self.llm.transformer.wte(tokens.input_ids)
    combined = torch.cat([prefix, text_embeds], dim=1)
    mask = torch.cat([torch.ones(batch_size, self.num_prefix_tokens, device=device), tokens.attention_mask], dim=1)

    labels = tokens.input_ids.clone()
    for i in range(batch_size):
        labels[i, :q_lengths[i]] = -100
    labels = torch.cat([torch.full((batch_size, self.num_prefix_tokens), -100, dtype=torch.long, device=device), labels], dim=1)

    return self.llm(inputs_embeds=combined, attention_mask=mask, labels=labels).loss

def _adapter_compute_numerical_loss(self, physics_states, object_mask, targets):
    preds = self.predict_numerical(physics_states, object_mask)
    total_loss, count = 0.0, 0
    for key in targets:
        if key in preds:
            total_loss += F.mse_loss(preds[key], targets[key].to(preds[key].device))
            count += 1
    return total_loss / max(count, 1)

def _adapter_answer_descriptive(self, physics_states, object_mask, question_text):
    features = self.extract_physics_features(physics_states, object_mask)
    return self.descriptive_head.predict_batch(features, question_text)

def _adapter_compute_descriptive_loss(self, physics_states, object_mask, question_text, answer_text, label_smoothing=0.1):
    features = self.extract_physics_features(physics_states, object_mask)
    return self.descriptive_head.compute_loss(features, question_text, answer_text, label_smoothing)

def _adapter_score_answer_candidates(self, physics_states, object_mask, question_text, answer_candidates, max_length=128):
    batch_size = physics_states.size(0)
    device = physics_states.device
    k = len(answer_candidates[0])

    features = self.extract_physics_features(physics_states, object_mask)
    prefix = self.create_prefix_tokens(features)
    expanded_prefix = prefix.unsqueeze(1).expand(-1, k, -1, -1).reshape(batch_size * k, self.num_prefix_tokens, self.llm_dim)

    expanded_questions, expanded_full = [], []
    for q, choices in zip(question_text, answer_candidates):
        for a in choices:
            expanded_questions.append(q)
            expanded_full.append(q + " Answer: " + a)

    tokens = self.tokenizer(expanded_full, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
    q_lengths = self._compute_question_lengths(expanded_questions)

    text_embeds = self.llm.transformer.wte(tokens.input_ids)
    combined = torch.cat([expanded_prefix, text_embeds], dim=1)
    mask = torch.cat([torch.ones(batch_size * k, self.num_prefix_tokens, device=device), tokens.attention_mask], dim=1)

    labels = tokens.input_ids.clone()
    for i in range(batch_size * k):
        labels[i, :q_lengths[i]] = -100
    labels = torch.cat([torch.full((batch_size * k, self.num_prefix_tokens), -100, dtype=torch.long, device=device), labels], dim=1)

    outputs = self.llm(inputs_embeds=combined, attention_mask=mask, labels=labels, return_dict=True)
    logits = outputs.logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    per_token_loss = F.cross_entropy(logits.view(-1, logits.size(-1)), shift_labels.view(-1), reduction='none').view(batch_size * k, -1)
    valid_mask = (shift_labels != -100).float()
    per_sample_loss = (per_token_loss * valid_mask).sum(dim=-1) / (valid_mask.sum(dim=-1) + 1e-8)

    return (-per_sample_loss).reshape(batch_size, k)

def _adapter_select_answer_from_choices(self, physics_states, object_mask, question_text, choices, max_length=128):
    scores = self.score_answer_candidates(physics_states, object_mask, question_text, choices, max_length)
    best = torch.argmax(scores, dim=-1).tolist()
    return [c[i] for c, i in zip(choices, best)]

def _adapter_compute_contrastive_loss(self, physics_states, object_mask, temperature=0.07):
    batch_size = physics_states.size(0)
    if batch_size < 1:
        return torch.tensor(0.0, device=physics_states.device)

    real_features = self.extract_physics_features(physics_states, object_mask)
    zero_features = self.extract_physics_features(torch.zeros_like(physics_states), object_mask)

    real_prefix = self.create_prefix_tokens(real_features).view(batch_size, -1)
    zero_prefix = self.create_prefix_tokens(zero_features).view(batch_size, -1)

    real_norm = F.normalize(real_prefix, p=2, dim=1)
    zero_norm = F.normalize(zero_prefix, p=2, dim=1)
    cos_sim = (real_norm * zero_norm).sum(dim=1)

    return F.relu(cos_sim - 0.5).mean()

def _adapter_compute_multiple_choice_loss(self, physics_states, object_mask, question_text, choices,
                                          correct_choice_idx, max_length=128, label_smoothing=0.1):
    """Compute MCQ loss using score_answer_candidates."""
    scores = self.score_answer_candidates(
        physics_states=physics_states,
        object_mask=object_mask,
        question_text=question_text,
        answer_candidates=choices,
        max_length=max_length
    )
    return F.cross_entropy(scores, correct_choice_idx.to(scores.device), label_smoothing=label_smoothing)

def _adapter_compute_combined_loss(self, physics_states, object_mask, questions, answers,
                                    choices=None, correct_choice_idx=None, numerical_targets=None,
                                    categorical_weight=1.0, numerical_weight=1.0, descriptive_weight=1.0):
    """
    Compute combined loss for categorical, descriptive, and numerical outputs.

    Routes questions to appropriate loss function:
    - Descriptive questions -> descriptive_head loss
    - MCQ questions -> multiple_choice_loss
    - Other questions -> LLM loss
    """
    device = physics_states.device

    # Separate descriptive vs non-descriptive questions
    desc_indices = []
    non_desc_indices = []
    for i, q in enumerate(questions):
        cat = classify_clevrer_question(q)
        if cat == CLEVRERQuestionCategory.DESCRIPTIVE:
            desc_indices.append(i)
        else:
            non_desc_indices.append(i)

    # Compute descriptive loss if any descriptive questions
    desc_loss = torch.tensor(0.0, device=device)
    if desc_indices:
        desc_questions = [questions[i] for i in desc_indices]
        desc_answers = [answers[i] for i in desc_indices]
        desc_states = physics_states[desc_indices]
        desc_masks = object_mask[desc_indices]
        desc_loss = self.compute_descriptive_loss(
            desc_states, desc_masks, desc_questions, desc_answers
        )

    # Compute categorical loss for non-descriptive questions
    cat_loss = torch.tensor(0.0, device=device)
    if non_desc_indices:
        non_desc_questions = [questions[i] for i in non_desc_indices]
        non_desc_answers = [answers[i] for i in non_desc_indices]
        non_desc_states = physics_states[non_desc_indices]
        non_desc_masks = object_mask[non_desc_indices]

        # Check if these have MCQ choices
        if choices is not None and correct_choice_idx is not None:
            non_desc_choices = [choices[i] for i in non_desc_indices]
            non_desc_correct = correct_choice_idx[non_desc_indices]
            cat_loss = self.compute_multiple_choice_loss(
                physics_states=non_desc_states,
                object_mask=non_desc_masks,
                question_text=non_desc_questions,
                choices=non_desc_choices,
                correct_choice_idx=non_desc_correct
            )
        else:
            cat_loss = self.compute_loss(
                non_desc_states, non_desc_masks, non_desc_questions, non_desc_answers
            )

    # Compute numerical loss
    if numerical_targets is not None:
        num_loss = self.compute_numerical_loss(physics_states, object_mask, numerical_targets)
    else:
        num_loss = torch.tensor(0.0, device=device)

    total_loss = (categorical_weight * cat_loss +
                  descriptive_weight * desc_loss +
                  numerical_weight * num_loss)

    return total_loss, {
        'categorical': cat_loss,
        'descriptive': desc_loss,
        'numerical': num_loss
    }

def _adapter_compute_combined_loss_with_contrastive(self, physics_states, object_mask, questions, answers,
                                                     choices=None, correct_choice_idx=None, numerical_targets=None,
                                                     categorical_weight=1.0, numerical_weight=1.0, descriptive_weight=1.0,
                                                     contrastive_weight=0.1):
    """Compute combined loss with contrastive term to prevent modality collapse."""
    total_loss, loss_dict = self.compute_combined_loss(
        physics_states, object_mask, questions, answers,
        choices=choices, correct_choice_idx=correct_choice_idx,
        numerical_targets=numerical_targets,
        categorical_weight=categorical_weight,
        numerical_weight=numerical_weight,
        descriptive_weight=descriptive_weight
    )

    # Add contrastive loss
    contr_loss = self.compute_contrastive_loss(physics_states, object_mask)
    loss_dict['contrastive'] = contr_loss
    total_loss = total_loss + contrastive_weight * contr_loss

    return total_loss, loss_dict

def _adapter_set_training_phase(self, phase):
    for model in [self.physics_model, self.adapter, self.numerical_head, self.descriptive_head, self.llm]:
        model.eval()
        for param in model.parameters():
            param.requires_grad = False

    if phase == 'adapter':
        self.adapter.train()
        self.numerical_head.train()
        self.descriptive_head.train()
        for m in [self.adapter, self.numerical_head, self.descriptive_head]:
            for param in m.parameters():
                param.requires_grad = True
    elif phase == 'llm_head':
        self.adapter.train()
        self.numerical_head.train()
        self.descriptive_head.train()
        self.llm.lm_head.train()
        for m in [self.adapter, self.numerical_head, self.descriptive_head]:
            for param in m.parameters():
                param.requires_grad = True
        for param in self.llm.lm_head.parameters():
            param.requires_grad = True
    elif phase in ['llm_full', 'full']:
        self.adapter.train()
        self.numerical_head.train()
        self.descriptive_head.train()
        self.llm.train()
        for m in [self.adapter, self.numerical_head, self.descriptive_head, self.llm]:
            for param in m.parameters():
                param.requires_grad = True
    print(f"[PHASE] {phase}")

# Attach methods to class
GroundedPhysicsLM.compute_loss = _adapter_compute_loss
GroundedPhysicsLM.compute_numerical_loss = _adapter_compute_numerical_loss
GroundedPhysicsLM.answer_descriptive = _adapter_answer_descriptive
GroundedPhysicsLM.compute_descriptive_loss = _adapter_compute_descriptive_loss
GroundedPhysicsLM.score_answer_candidates = _adapter_score_answer_candidates
GroundedPhysicsLM.select_answer_from_choices = _adapter_select_answer_from_choices
GroundedPhysicsLM.compute_contrastive_loss = _adapter_compute_contrastive_loss
GroundedPhysicsLM.compute_multiple_choice_loss = _adapter_compute_multiple_choice_loss
GroundedPhysicsLM.compute_combined_loss = _adapter_compute_combined_loss
GroundedPhysicsLM.compute_combined_loss_with_contrastive = _adapter_compute_combined_loss_with_contrastive
GroundedPhysicsLM.set_training_phase = _adapter_set_training_phase

def create_grounded_physics_lm(physics_model, **kwargs):
    """Factory function to create GroundedPhysicsLM."""
    return GroundedPhysicsLM(physics_model=physics_model, **kwargs)

print("GroundedPhysicsLM methods attached (including compute_combined_loss_with_contrastive)")

GroundedPhysicsLM methods attached (including compute_combined_loss_with_contrastive)


In [ ]:
# ============================================================
# CELL 9: LOAD PHYSICS MODEL CHECKPOINT (matching colab_train_adapter.ipynb)
# ============================================================

print("Loading PhysicsFormer checkpoint...")
print(f"  Checkpoint: {PHYSICS_CHECKPOINT}")

physics_checkpoint_path = Path(PHYSICS_CHECKPOINT)
if not physics_checkpoint_path.exists():
    raise FileNotFoundError(
        f"Physics checkpoint not found at: {physics_checkpoint_path}\n\n"
        f"Upload physics_former_best.pt to GDrive:\n"
        f"  {CHECKPOINT_PATH}/physics_former_best.pt"
    )

checkpoint = torch.load(physics_checkpoint_path, map_location=device, weights_only=False)

if 'model_state_dict' not in checkpoint:
    raise KeyError(f"Checkpoint missing 'model_state_dict' key. Keys: {list(checkpoint.keys())}")

model_state = checkpoint['model_state_dict']

# Handle _orig_mod prefix from torch.compile
has_orig_mod = any(k.startswith('_orig_mod.') for k in model_state.keys())
prefix = '_orig_mod.' if has_orig_mod else ''

if has_orig_mod:
    print("  Found _orig_mod. prefix (from torch.compile)")

# Detect architecture from checkpoint (matching colab_train_adapter.ipynb)
schema_key = f'{prefix}schema_classifier.3.bias'
num_schema_classes = model_state[schema_key].shape[0] if schema_key in model_state else 37

hidden_dim_key = f'{prefix}transformer_layers.0.attention.q_proj.weight'
hidden_dim = model_state[hidden_dim_key].shape[0] if hidden_dim_key in model_state else 768

num_layers = sum(1 for k in model_state.keys() 
                 if f'{prefix}transformer_layers.' in k and '.attention.q_proj.weight' in k)

num_heads_key = f'{prefix}transformer_layers.0.attention.attention_bias_net.2.bias'
num_heads = model_state[num_heads_key].shape[0] if num_heads_key in model_state else 24

# state_dim is auto-detected from config (should be 35 for physics_former_best.pt)
# The config.state_dim from TrainingConfig is 35
state_dim = 35  # This matches physics_former_best.pt

print(f"\n  Detected architecture:")
print(f"    state_dim: {state_dim}")
print(f"    hidden_dim: {hidden_dim}")
print(f"    num_layers: {num_layers}")
print(f"    num_heads: {num_heads}")
print(f"    schema_classes: {num_schema_classes}")

# Clean state dict
if has_orig_mod:
    cleaned_state = {k.replace('_orig_mod.', ''): v for k, v in model_state.items()}
else:
    cleaned_state = model_state

# Filter out RoPE cached tensors (will be rebuilt)
filtered_state = {k: v for k, v in cleaned_state.items()
                  if 'rope.cos_cached' not in k and 'rope.sin_cached' not in k}

print(f"\n  Checkpoint loaded: {len(filtered_state)} parameters")

In [ ]:
# ============================================================
# CELL 10: CREATE PHYSICS MODEL (matching colab_train_adapter.ipynb)
# ============================================================
# Uses FullPhysicsFormer with detected architecture parameters

from models.physics_former_full import FullPhysicsFormer
from configs.config import TrainingConfig

config = TrainingConfig()

# Create FullPhysicsFormer using detected architecture
physics_model = FullPhysicsFormer(
    state_dim=config.state_dim,  # 35
    hidden_dim=hidden_dim,       # 768 (detected)
    num_layers=num_layers,       # 8 (detected)
    num_heads=num_heads,         # 24 (detected)
    ff_dim=hidden_dim * 4,       # 3072
    max_objects=config.max_objects,  # 20
    dropout=config.dropout,      # 0.1
    num_schema_classes=num_schema_classes  # 37 (detected)
).to(device)

# Load checkpoint weights
missing_keys, unexpected_keys = physics_model.load_state_dict(filtered_state, strict=False)

if missing_keys:
    print(f"Missing keys (expected for new heads): {len(missing_keys)}")
    for k in missing_keys[:5]:
        print(f"  - {k}")
    if len(missing_keys) > 5:
        print(f"  ... and {len(missing_keys) - 5} more")

if unexpected_keys:
    print(f"Unexpected keys (from checkpoint): {len(unexpected_keys)}")
    for k in unexpected_keys[:5]:
        print(f"  - {k}")
    if len(unexpected_keys) > 5:
        print(f"  ... and {len(unexpected_keys) - 5} more")

# Freeze physics model for adapter training
physics_model.eval()
for param in physics_model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in physics_model.parameters())
print(f"\n✅ FullPhysicsFormer loaded: {total_params:,} parameters (frozen)")

In [11]:
# ============================================================
# CELL 11: CREATE GROUNDED PHYSICS LM
# ============================================================
# Updated to use hidden_dim detected from checkpoint (like working notebook)

print("Creating GroundedPhysicsLM with DistilGPT-2...")

model = create_grounded_physics_lm(
    physics_model=physics_model,
    physics_dim=hidden_dim,  # Use detected hidden_dim from checkpoint
    llm_name="distilgpt2",   # 82M params - edge-optimized
    num_prefix_tokens=NUM_PREFIX_TOKENS,
    freeze_physics=True,
    freeze_llm=True  # Will unfreeze progressively
).to(device)

# Alias for backward compatibility with training cells
adapter = model

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nGroundedPhysicsLM created:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Prefix tokens: {NUM_PREFIX_TOKENS}")
print(f"  Physics dim: {hidden_dim}")
print(f"  LLM: DistilGPT-2 (82M params)")

Creating GroundedPhysicsLM with DistilGPT-2...
Loading LLM: distilgpt2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GroundedPhysicsLM: LLM=distilgpt2, physics_dim=256, prefix_tokens=64

GroundedPhysicsLM created:
  Total parameters: 126,672,459
  Trainable parameters: 39,951,643
  Prefix tokens: 64
  Physics dim: 256
  LLM: DistilGPT-2 (82M params)


In [12]:
# ============================================================
# CELL 12: LOAD HDF5 DATA - Focus on Causal/Counterfactual Questions
# ============================================================
# Updated to use HDF5 format like colab_train_adapter.ipynb (working code)

import random
from torch.utils.data import Dataset, DataLoader

print(f'Loading CLEVRER HDF5 data from {HDF5_DATA_PATH}...')

# Check if HDF5 file exists
if not os.path.exists(HDF5_DATA_PATH):
    raise FileNotFoundError(
        f"HDF5 data not found at: {HDF5_DATA_PATH}\n\n"
        f"Upload clevrer_training_expanded.h5 to GDrive:\n"
        f"  {DATA_PATH}/clevrer_training_expanded.h5"
    )

# HDF5 Dataset class with question type filtering and choice shuffling
class CLEVRERHDF5Dataset(Dataset):
    """Dataset for loading CLEVRER training data from HDF5 format.

    Features (from working colab_train_adapter.ipynb):
    - Extracts MCQ choices from metadata for cross-entropy training
    - Shuffles choice order to prevent position bias
    - Filters by question type (causal reasoning focus)
    """

    def __init__(self, hdf5_path, max_samples=None, question_types=None, shuffle_choices=True):
        self.hdf5_path = hdf5_path
        self.hf = h5py.File(hdf5_path, 'r')
        self.shuffle_choices = shuffle_choices

        self.num_samples = self.hf.attrs['num_samples']
        self.seq_len = self.hf.attrs['seq_len']
        self.num_objects = self.hf.attrs['num_objects']
        self.state_dim = self.hf.attrs['state_dim']

        # Build filtered index by question type
        if question_types:
            self.indices = []
            for i in range(self.num_samples):
                qt = self.hf['question_types'][i]
                if isinstance(qt, bytes):
                    qt = qt.decode('utf-8')
                if any(t in qt.lower() for t in question_types):
                    self.indices.append(i)
            print(f'  Found {len(self.indices):,} samples matching types: {question_types}')
        else:
            self.indices = list(range(self.num_samples))
            print(f'  Found {len(self.indices):,} samples (all types)')

        # Apply max_samples limit
        if max_samples and len(self.indices) > max_samples:
            random.shuffle(self.indices)
            self.indices = self.indices[:max_samples]
            print(f'  Limited to {len(self.indices):,} samples')

        print(f'  State shape: ({self.seq_len}, {self.num_objects}, {self.state_dim})')
        print(f'  Choice shuffling: {shuffle_choices}')

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]

        # Load states and mask
        states = torch.from_numpy(self.hf['states'][real_idx].astype(np.float32))
        mask = torch.from_numpy(self.hf['masks'][real_idx].astype(np.float32))

        # Handle state dimensions - need [num_objects, state_dim] for model
        if states.dim() == 3:
            # If [seq_len, num_objects, state_dim], take last timestep
            states = states[-1]

        # Ensure correct dimensions
        if states.shape[0] != MAX_OBJECTS:
            if states.shape[0] < MAX_OBJECTS:
                pad = torch.zeros(MAX_OBJECTS - states.shape[0], states.shape[1])
                states = torch.cat([states, pad], dim=0)
            else:
                states = states[:MAX_OBJECTS]

        if states.shape[1] != STATE_DIM:
            if states.shape[1] < STATE_DIM:
                pad = torch.zeros(states.shape[0], STATE_DIM - states.shape[1])
                states = torch.cat([states, pad], dim=1)
            else:
                states = states[:, :STATE_DIM]

        # Handle mask dimensions
        if mask.dim() == 2:
            mask = mask[-1]  # Take last timestep
        if len(mask) != MAX_OBJECTS:
            if len(mask) < MAX_OBJECTS:
                mask = torch.cat([mask, torch.zeros(MAX_OBJECTS - len(mask))])
            else:
                mask = mask[:MAX_OBJECTS]

        # Load text fields
        question = self.hf['questions'][real_idx]
        if isinstance(question, bytes):
            question = question.decode('utf-8')

        answer = self.hf['answers'][real_idx]
        if isinstance(answer, bytes):
            answer = answer.decode('utf-8')

        question_type = self.hf['question_types'][real_idx]
        if isinstance(question_type, bytes):
            question_type = question_type.decode('utf-8')

        # Load metadata for MCQ choices
        metadata_str = self.hf['metadata'][real_idx]
        if isinstance(metadata_str, bytes):
            metadata_str = metadata_str.decode('utf-8')
        metadata = json.loads(metadata_str)

        # Load numerical targets
        numerical_targets = self.hf['numerical_targets'][real_idx]

        # Extract MCQ choices from metadata for proper cross-entropy training
        choices = None
        correct_choice_idx = None
        if 'choices' in metadata and isinstance(metadata['choices'], list):
            choice_data = metadata['choices']

            # Shuffle choices to prevent position bias
            if self.shuffle_choices:
                indices = list(range(len(choice_data)))
                random.shuffle(indices)
                choice_data = [choice_data[i] for i in indices]

            choices = [c['choice'] for c in choice_data]
            # Find the correct choice index (after shuffling)
            for i, c in enumerate(choice_data):
                if c.get('answer') == 'correct':
                    correct_choice_idx = i
                    break

        return {
            'physics_states': states,
            'object_mask': mask,
            'questions': question,
            'answers': answer,
            'question_type': question_type,
            'metadata': metadata,
            'choices': choices,
            'correct_choice_idx': correct_choice_idx,
            'numerical_targets': torch.tensor([
                float(numerical_targets[0]),  # count
                float(numerical_targets[1]),  # value
                0.0, 0.0, 0.0, 0.0  # padding for 6 outputs
            ], dtype=torch.float32)
        }

    def close(self):
        self.hf.close()


# Collate function for batching (from working notebook)
def collate_fn(batch):
    """Collate function that handles variable-length choices."""
    physics_states = torch.stack([b['physics_states'] for b in batch])
    object_mask = torch.stack([b['object_mask'] for b in batch])
    questions = [b['questions'] for b in batch]
    answers = [b['answers'] for b in batch]
    numerical_targets = torch.stack([b['numerical_targets'] for b in batch])

    # Handle choices (may be None for some samples)
    choices = [b['choices'] for b in batch]
    correct_choice_idx = [b['correct_choice_idx'] for b in batch]

    # Convert correct_choice_idx to tensor if all are valid
    if all(idx is not None for idx in correct_choice_idx):
        correct_choice_idx = torch.tensor(correct_choice_idx, dtype=torch.long)
    else:
        correct_choice_idx = None

    return {
        'physics_states': physics_states,
        'object_mask': object_mask,
        'questions': questions,
        'answers': answers,
        'numerical_targets': numerical_targets,
        'choices': choices,
        'correct_choice_idx': correct_choice_idx
    }


# Load dataset with focus on causal reasoning questions
full_dataset = CLEVRERHDF5Dataset(
    HDF5_DATA_PATH,
    max_samples=MAX_TRAIN_SAMPLES,
    question_types=FOCUS_QUESTION_TYPES,
    shuffle_choices=True  # Prevent position bias
)

# Create train/test split (90/10)
total_size = len(full_dataset)
train_size = int(0.9 * total_size)

train_indices = list(range(train_size))
test_indices = list(range(train_size, total_size))

random.seed(42)
random.shuffle(train_indices)

class SubsetDataset(Dataset):
    def __init__(self, base_dataset, indices):
        self.base = base_dataset
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        return self.base[self.indices[idx]]

train_dataset = SubsetDataset(full_dataset, train_indices)
test_dataset = SubsetDataset(full_dataset, test_indices)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print(f'\n✅ Data loaded (causal reasoning focus):')
print(f'   Train: {len(train_dataset):,} samples ({len(train_loader):,} batches)')
print(f'   Test: {len(test_dataset):,} samples ({len(test_loader):,} batches)')

# Show question type distribution
from collections import Counter
qtypes = Counter()
for i in range(min(500, len(full_dataset))):
    qtypes[full_dataset[i]['question_type']] += 1
print(f'\nQuestion type distribution (sample of 500):')
for qt, count in sorted(qtypes.items()):
    print(f'  {qt}: {count}')

Loading CLEVRER HDF5 data from /content/drive/MyDrive/physics_action_predictor/data/clevrer_training_expanded.h5...


FileNotFoundError: HDF5 data not found at: /content/drive/MyDrive/physics_action_predictor/data/clevrer_training_expanded.h5

Upload clevrer_training_expanded.h5 to GDrive:
  /content/drive/MyDrive/physics_action_predictor/data/clevrer_training_expanded.h5

In [ ]:
# ============================================================
# CELL 13: TRAINING FUNCTIONS WITH PHYSICS V2 TECHNOLOGIES
# ============================================================
# Features from train_physics_former_v2.ipynb:
# - AMP (Automatic Mixed Precision) for faster training
# - OneCycleLR scheduler for better convergence
# - PlateauTracker + Catapult for plateau breakthrough
# - Diversity penalty / contrastive loss (model's compute_combined_loss_with_contrastive)
# - Physics usage validation
# - Catapult resets early stopping patience (plateau epochs don't count against progression)
# - SAMPLE DISPLAY: Shows actual questions and model answers for validation

from torch.optim.lr_scheduler import OneCycleLR

# ============================================================
# AMP CONFIGURATION
# ============================================================
USE_AMP = torch.cuda.is_available()  # Only use AMP on CUDA

if USE_AMP:
    scaler = torch.cuda.amp.GradScaler()
    print("AMP: ENABLED (GradScaler initialized)")
else:
    scaler = None
    print("AMP: DISABLED (CPU mode)")

# Physics validation configuration
VALIDATE_PHYSICS_EVERY = 3  # Validate physics usage every N epochs
PHYSICS_SIM_WARNING_THRESHOLD = 0.95  # Warn if cosine similarity > this (model ignoring physics)


# ============================================================
# PLATEAU BREAKTHROUGH (from physics former v2)
# ============================================================
class PlateauTracker:
    """
    Detects training plateaus and triggers learning rate catapults.
    Based on 2025 research showing transformers learn in "bursts" after plateaus.

    Key behavior: When catapult triggers, it signals to reset early stopping patience.
    This gives the catapult a fair chance to work before stopping training.
    """
    def __init__(self, patience: int = 5, min_delta: float = 0.003):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.plateau_count = 0
        self.catapult_countdown = 0
        self.catapult_lr = None
        self.base_lr = None
        self.catapult_just_triggered = False  # Flag for resetting early stopping

    def update(self, loss: float) -> bool:
        """Returns True if plateau detected."""
        if loss < self.best_loss - self.min_delta:
            self.best_loss = loss
            self.plateau_count = 0
            return False
        else:
            self.plateau_count += 1
            return self.plateau_count >= self.patience

    def trigger_catapult(self, optimizer, multiplier: float = 1.5, duration: int = 3):
        """Trigger LR catapult to escape plateau."""
        self.base_lr = optimizer.param_groups[0]['lr']
        config_base_lr = PHASE1_LR  # Use phase 1 LR as base
        self.catapult_lr = config_base_lr * multiplier
        self.catapult_countdown = duration
        self.catapult_just_triggered = True  # Signal to reset early stopping
        self.plateau_count = 0  # Reset plateau counter

        for param_group in optimizer.param_groups:
            param_group['lr'] = self.catapult_lr
        print(f"    🚀 CATAPULT: LR boosted to {self.catapult_lr:.6f} for {duration} epochs")
        print(f"    🚀 CATAPULT: Early stopping patience RESET (giving catapult a fair chance)")

    def enforce_catapult_lr(self, optimizer):
        """Call AFTER scheduler.step() to override with catapult LR."""
        if self.catapult_countdown > 0 and self.catapult_lr is not None:
            for param_group in optimizer.param_groups:
                param_group['lr'] = self.catapult_lr
            self.catapult_countdown -= 1
            if self.catapult_countdown == 0:
                print(f"    🚀 CATAPULT: Ended, returning to scheduler LR")
                self.catapult_lr = None

    def should_reset_early_stopping(self) -> bool:
        """Check and consume the catapult trigger flag."""
        if self.catapult_just_triggered:
            self.catapult_just_triggered = False
            return True
        return False

    def is_in_catapult(self) -> bool:
        """Check if currently in catapult period (don't count against early stopping)."""
        return self.catapult_countdown > 0


# ============================================================
# PHYSICS USAGE VALIDATION
# ============================================================
def validate_physics_usage(model, val_loader, device, num_samples=20):
    """
    Validate that the model is actually using physics information.

    Compares prefix tokens between real physics and zero physics inputs.
    Returns cosine similarity (lower = better physics usage).

    If similarity > 0.95, the model is likely ignoring physics input!
    """
    model.eval()
    similarities = []
    differences = []

    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            if i >= num_samples:
                break

            states = batch['physics_states'].to(device)
            masks = batch['object_mask'].to(device)
            zero_states = torch.zeros_like(states)

            # Get prefix tokens for real and zero physics
            real_features = model.extract_physics_features(states, masks)
            zero_features = model.extract_physics_features(zero_states, masks)

            real_prefix = model.create_prefix_tokens(real_features)
            zero_prefix = model.create_prefix_tokens(zero_features)

            # Flatten and compute cosine similarity
            real_flat = real_prefix.view(real_prefix.size(0), -1)
            zero_flat = zero_prefix.view(zero_prefix.size(0), -1)

            cos_sim = torch.nn.functional.cosine_similarity(real_flat, zero_flat, dim=1)
            diff_norm = (real_flat - zero_flat).norm(dim=1)

            similarities.extend(cos_sim.cpu().tolist())
            differences.extend(diff_norm.cpu().tolist())

    avg_similarity = sum(similarities) / len(similarities) if similarities else 1.0
    avg_difference = sum(differences) / len(differences) if differences else 0.0

    return {
        'avg_cosine_similarity': avg_similarity,
        'avg_difference_norm': avg_difference,
        'num_samples': len(similarities)
    }


# ============================================================
# SAMPLE DISPLAY - Show actual Q&A for validation
# ============================================================
def display_validation_samples_mcq(model, val_loader, device, num_samples=5):
    """
    Display actual questions and model answers to verify the model is working.
    Shows question, expected answer, model prediction, and correctness.
    """
    model.eval()
    print("\n" + "=" * 70)
    print("VALIDATION SAMPLES - Actual Questions and Answers")
    print("=" * 70)

    samples_shown = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in val_loader:
            if samples_shown >= num_samples:
                break

            physics_states = batch['physics_states'].to(device)
            object_mask = batch['object_mask'].to(device)
            questions = batch['questions']
            answers = batch['answers']

            # Get model predictions
            predictions = model(physics_states, object_mask, questions)

            # Also get descriptive head predictions for comparison
            features = model.extract_physics_features(physics_states, object_mask)
            desc_preds = model.descriptive_head.predict_batch(features, questions)

            for i, (q, expected, pred, desc_pred) in enumerate(zip(questions, answers, predictions, desc_preds)):
                if samples_shown >= num_samples:
                    break

                # Clean predictions
                pred_clean = pred.split('\n')[0].strip().lower()
                exp_clean = expected.strip().lower()

                # Check correctness
                # Don't accept empty predictions as correct
                is_correct = (
                    len(pred_clean) > 0 and (
                        pred_clean == exp_clean or
                        exp_clean in pred_clean or
                        (len(pred_clean) > 3 and pred_clean in exp_clean)
                    )
                )

                status = "✓ CORRECT" if is_correct else "✗ WRONG"
                if is_correct:
                    correct += 1
                total += 1

                print(f"\n[Sample {samples_shown + 1}] {status}")
                print(f"  Q: {q[:80]}{'...' if len(q) > 80 else ''}")
                print(f"  Expected: {expected}")
                print(f"  LLM Pred: {pred[:60]}{'...' if len(pred) > 60 else ''}")
                print(f"  Desc Head: {desc_pred}")

                samples_shown += 1

    accuracy = 100.0 * correct / max(total, 1)
    print(f"\n{'=' * 70}")
    print(f"Sample Accuracy: {correct}/{total} ({accuracy:.1f}%)")
    print("=" * 70)
    return accuracy


# ============================================================
# TRAINING EPOCH WITH AMP + MODEL'S COMPUTE_COMBINED_LOSS_WITH_CONTRASTIVE
# ============================================================
def _safe_item(x):
    """Safely get scalar value from tensor or float."""
    return x.item() if hasattr(x, 'item') else x

def train_epoch_v2(adapter, dataloader, optimizer, scheduler, device,
                   use_contrastive=True, contrastive_weight=0.1,
                   use_amp=True, scaler=None, plateau_tracker=None):
    """
    Train for one epoch using model's built-in compute_combined_loss_with_contrastive.

    Features:
    - AMP (Automatic Mixed Precision) for faster training on GPU
    - OneCycleLR scheduler support
    - PlateauTracker integration for catapult
    - Uses adapter.compute_combined_loss_with_contrastive()
    """
    adapter.train()
    total_loss = 0.0
    cat_loss_sum = 0.0
    num_loss_sum = 0.0
    desc_loss_sum = 0.0
    contr_loss_sum = 0.0
    num_batches = 0
    grad_norm_sum = 0.0

    pbar = tqdm(dataloader, desc="Training", leave=False)
    for batch in pbar:
        physics_states = batch['physics_states'].to(device)
        object_mask = batch['object_mask'].to(device)
        questions = batch['questions']
        answers = batch['answers']
        choices = batch.get('choices')
        correct_choice_idx = batch.get('correct_choice_idx')
        if correct_choice_idx is not None:
            correct_choice_idx = correct_choice_idx.to(device)
        numerical_targets = batch['numerical_targets'].to(device)

        optimizer.zero_grad()

        # Use AMP autocast if enabled
        if use_amp and scaler is not None:
            with torch.cuda.amp.autocast():
                if use_contrastive:
                    loss, loss_dict = adapter.compute_combined_loss_with_contrastive(
                        physics_states, object_mask, questions, answers,
                        choices=choices,
                        correct_choice_idx=correct_choice_idx,
                        numerical_targets=numerical_targets,
                        categorical_weight=1.0,
                        numerical_weight=0.5,
                        contrastive_weight=contrastive_weight
                    )
                else:
                    loss, loss_dict = adapter.compute_combined_loss(
                        physics_states, object_mask, questions, answers,
                        choices=choices,
                        correct_choice_idx=correct_choice_idx,
                        numerical_targets=numerical_targets,
                        categorical_weight=1.0,
                        numerical_weight=0.5
                    )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
            grad_norm_sum += grad_norm.item() if hasattr(grad_norm, "item") else grad_norm
            scaler.step(optimizer)
            scaler.update()
        else:
            # Non-AMP path
            if use_contrastive:
                loss, loss_dict = adapter.compute_combined_loss_with_contrastive(
                    physics_states, object_mask, questions, answers,
                    choices=choices,
                    correct_choice_idx=correct_choice_idx,
                    numerical_targets=numerical_targets,
                    categorical_weight=1.0,
                    numerical_weight=0.5,
                    contrastive_weight=contrastive_weight
                )
            else:
                loss, loss_dict = adapter.compute_combined_loss(
                    physics_states, object_mask, questions, answers,
                    choices=choices,
                    correct_choice_idx=correct_choice_idx,
                    numerical_targets=numerical_targets,
                    categorical_weight=1.0,
                    numerical_weight=0.5
                )

            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
            grad_norm_sum += grad_norm.item() if hasattr(grad_norm, "item") else grad_norm
            optimizer.step()

        # Step scheduler (OneCycleLR steps per batch)
        if scheduler is not None:
            scheduler.step()
            # Enforce catapult LR if active
            if plateau_tracker is not None:
                plateau_tracker.enforce_catapult_lr(optimizer)

        # Accumulate losses
        total_loss += loss.item()
        cat_loss_sum += _safe_item(loss_dict['categorical']) if hasattr(loss_dict['categorical'], 'item') else loss_dict['categorical']
        num_loss_sum += _safe_item(loss_dict['numerical']) if hasattr(loss_dict['numerical'], 'item') else loss_dict['numerical']
        desc_loss = loss_dict.get('descriptive', torch.tensor(0.0)).item()
        desc_loss_sum += desc_loss
        contr_loss = loss_dict.get('contrastive', torch.tensor(0.0)).item()
        contr_loss_sum += contr_loss
        num_batches += 1

        # Update progress bar with detailed loss breakdown
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'cat': f'{_safe_item(loss_dict["categorical"]):.3f}',
            'num': f'{_safe_item(loss_dict["numerical"]):.3f}',
            'contr': f'{contr_loss:.3f}'
        })

    return {
        'total_loss': total_loss / max(num_batches, 1),
        'categorical_loss': cat_loss_sum / max(num_batches, 1),
        'numerical_loss': num_loss_sum / max(num_batches, 1),
        'descriptive_loss': desc_loss_sum / max(num_batches, 1),
        'contrastive_loss': contr_loss_sum / max(num_batches, 1),
        'num_batches': num_batches,
        'grad_norm': grad_norm_sum / max(num_batches, 1)
    }


# ============================================================
# TRAINING PHASE WITH ALL V2 TECHNOLOGIES
# ============================================================
def train_phase_v2(adapter, train_loader, test_loader, optimizer, device,
                   phase_name, max_epochs=100, patience=5, min_delta=0.001,
                   checkpoint_dir=None, phase_num=1, save_every_n_epochs=3,
                   use_contrastive=True, contrastive_weight=0.1,
                   validate_every_n_epochs=5, physics_sim_threshold=0.95,
                   use_amp=True, scaler=None, use_plateau_tracker=True,
                   show_samples_every_n_epochs=3):
    """
    Train with all Physics Former V2 technologies:
    - AMP (Automatic Mixed Precision)
    - OneCycleLR scheduler (per-epoch)
    - PlateauTracker + Catapult (with patience reset)
    - Physics usage validation
    - Model's compute_combined_loss_with_contrastive
    - SAMPLE DISPLAY: Shows actual Q&A every N epochs

    Key: Plateau epochs DON'T count against level progression.
    When catapult triggers, early stopping patience is reset.
    """
    print(f"\n{'=' * 70}")
    print(f"TRAINING PHASE: {phase_name}")
    print(f"{'=' * 70}")
    print(f"  Early stopping: patience={patience}, min_delta={min_delta}")
    print(f"  Contrastive loss: {'ENABLED' if use_contrastive else 'DISABLED'} (weight={contrastive_weight})")
    print(f"  AMP: {'ENABLED' if use_amp and scaler else 'DISABLED'}")
    print(f"  OneCycleLR: Per-epoch scheduler")
    print(f"  PlateauTracker: {'ENABLED' if use_plateau_tracker else 'DISABLED'} (resets early stopping on catapult)")
    print(f"  Physics validation: every {validate_every_n_epochs} epochs (warn if sim > {physics_sim_threshold})")
    print(f"  Sample display: every {show_samples_every_n_epochs} epochs")
    if checkpoint_dir:
        print(f"  Checkpoints: {checkpoint_dir}")
    print("=" * 70)

    # Show trainable vs frozen parameters
    trainable_params = sum(p.numel() for p in adapter.parameters() if p.requires_grad)
    frozen_params = sum(p.numel() for p in adapter.parameters() if not p.requires_grad)
    total_params = trainable_params + frozen_params
    print(f"\n  Parameters: {trainable_params:,} trainable / {frozen_params:,} frozen ({100*trainable_params/total_params:.1f}% active)")
    print("=" * 70)

    best_loss = float('inf')
    epochs_without_improvement = 0
    plateau_tracker = PlateauTracker(patience=3, min_delta=0.003) if use_plateau_tracker else None

    for epoch in range(max_epochs):
        # Create OneCycleLR scheduler for this epoch
        scheduler = OneCycleLR(
            optimizer,
            max_lr=optimizer.param_groups[0]['lr'] * 2,  # Peak at 2x current LR
            steps_per_epoch=len(train_loader),
            epochs=1
        )

        # Training
        loss_dict = train_epoch_v2(
            adapter, train_loader, optimizer, scheduler, device,
            use_contrastive=use_contrastive,
            contrastive_weight=contrastive_weight,
            use_amp=use_amp,
            scaler=scaler,
            plateau_tracker=plateau_tracker
        )

        avg_loss = loss_dict['total_loss']

        # Check for plateau and trigger catapult
        catapult_triggered = False
        if plateau_tracker is not None:
            if plateau_tracker.update(avg_loss):
                plateau_tracker.trigger_catapult(optimizer, multiplier=1.5, duration=3)
                catapult_triggered = True

            # Reset early stopping patience when catapult triggers
            if plateau_tracker.should_reset_early_stopping():
                epochs_without_improvement = 0
                print(f"    → Early stopping patience reset to 0/{patience}")

        # Check for improvement (only count if NOT in catapult period)
        improved = False
        in_catapult = plateau_tracker.is_in_catapult() if plateau_tracker else False

        if best_loss - avg_loss > min_delta:
            best_loss = avg_loss
            epochs_without_improvement = 0
            status = "improved ✓"
            improved = True
        elif in_catapult:
            # During catapult, don't increment early stopping counter
            status = f"catapult active 🚀 (not counting against patience)"
        else:
            epochs_without_improvement += 1
            status = f"no improvement ({epochs_without_improvement}/{patience})"

        # Log epoch results with detailed loss breakdown
        lr = optimizer.param_groups[0]['lr']
        print(f"\nEpoch {epoch+1:3d}/{max_epochs}")
        print(f"  Loss: {avg_loss:.4f} (best: {best_loss:.4f}) | {status}")
        print(f"  Breakdown: cat={loss_dict['categorical_loss']:.4f} | num={loss_dict['numerical_loss']:.4f} | "
              f"desc={loss_dict['descriptive_loss']:.4f} | contr={loss_dict['contrastive_loss']:.4f}")
        print(f"  LR: {lr:.2e} | Grad norm: {loss_dict.get('grad_norm', 0):.4f} | Patience: {epochs_without_improvement}/{patience}")

        # Show sample Q&A every N epochs
        if (epoch + 1) % show_samples_every_n_epochs == 0:
            display_validation_samples_mcq(adapter, test_loader, device, num_samples=5)

        # Physics usage validation every N epochs
        if (epoch + 1) % validate_every_n_epochs == 0:
            print("\n  [PHYSICS VALIDATION]")
            val_results = validate_physics_usage(adapter, test_loader, device)
            sim = val_results['avg_cosine_similarity']
            diff = val_results['avg_difference_norm']
            print(f"    Prefix cosine similarity (real vs zero): {sim:.4f}")
            print(f"    Prefix difference norm: {diff:.2f}")
            if sim > physics_sim_threshold:
                print(f"    ⚠️  WARNING: High similarity suggests model NOT using physics!")
            elif sim < 0.8:
                print(f"    ✓ GOOD: Low similarity suggests model IS using physics!")

        # Save checkpoint
        if checkpoint_dir and ((epoch + 1) % save_every_n_epochs == 0 or improved):
            ckpt_path = Path(checkpoint_dir) / f"adapter_phase{phase_num}_epoch{epoch+1}_loss{avg_loss:.4f}.pt"
            ckpt_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save({
                'model_state_dict': adapter.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'phase': phase_num,
                'epoch': epoch + 1,
                'loss': avg_loss,
                'best_loss': best_loss,
            }, ckpt_path)
            print(f"  → Checkpoint saved: {ckpt_path.name}")

        # Only trigger early stopping if not in catapult period
        if epochs_without_improvement >= patience and not in_catapult:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            break

    # Final physics validation and sample display
    print("\n" + "=" * 70)
    print("PHASE COMPLETE - FINAL VALIDATION")
    print("=" * 70)

    val_results = validate_physics_usage(adapter, test_loader, device)
    print(f"  Physics prefix cosine similarity: {val_results['avg_cosine_similarity']:.4f}")
    print(f"  Physics prefix difference norm: {val_results['avg_difference_norm']:.2f}")

    display_validation_samples_mcq(adapter, test_loader, device, num_samples=10)

    # Save phase-end checkpoint
    if checkpoint_dir:
        phase_end_path = Path(checkpoint_dir) / f"adapter_phase{phase_num}_complete_loss{best_loss:.4f}.pt"
        torch.save({
            'model_state_dict': adapter.state_dict(),
            'phase': phase_num,
            'loss': best_loss,
        }, phase_end_path)
        print(f"  → Phase complete checkpoint: {phase_end_path.name}")

    print(f"\n{phase_name} complete! Best loss: {best_loss:.4f}")
    return best_loss


# ============================================================
# EVALUATION WITH SAMPLE DISPLAY
# ============================================================
def evaluate(adapter, dataloader, device, num_samples=100, show_samples=True, num_display=10):
    """Evaluate adapter accuracy with optional sample display."""
    adapter.eval()
    correct = 0
    total = 0
    samples_to_show = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            if total >= num_samples:
                break

            physics_states = batch['physics_states'].to(device)
            object_mask = batch['object_mask'].to(device)
            questions = batch['questions']
            answers = batch['answers']

            predictions = adapter(physics_states, object_mask, questions)

            for pred, expected, q in zip(predictions, answers, questions):
                pred_clean = pred.split('\n')[0].strip().lower()
                exp_clean = expected.strip().lower()

                # Don't accept empty predictions as correct
                is_correct = (
                    len(pred_clean) > 0 and (
                        pred_clean == exp_clean or
                        exp_clean in pred_clean or
                        (len(pred_clean) > 3 and pred_clean in exp_clean)
                    )
                )

                if is_correct:
                    correct += 1

                if len(samples_to_show) < num_display:
                    samples_to_show.append({
                        'question': q,
                        'expected': expected,
                        'predicted': pred,
                        'correct': is_correct
                    })

                total += 1

    accuracy = 100.0 * correct / max(total, 1)

    if show_samples and samples_to_show:
        print("\n" + "=" * 70)
        print("EVALUATION SAMPLES")
        print("=" * 70)
        for i, s in enumerate(samples_to_show):
            status = "✓" if s['correct'] else "✗"
            print(f"\n[{i+1}] {status}")
            print(f"  Q: {s['question'][:70]}{'...' if len(s['question']) > 70 else ''}")
            print(f"  Expected: {s['expected']}")
            print(f"  Predicted: {s['predicted'][:50]}{'...' if len(s['predicted']) > 50 else ''}")
        print("=" * 70)

    return accuracy


print("\n" + "=" * 70)
print("TRAINING TECHNOLOGIES LOADED (Physics Former V2)")
print("=" * 70)
print("  ✓ AMP (Automatic Mixed Precision) - 2x faster on GPU")
print("  ✓ OneCycleLR scheduler - Better convergence")
print("  ✓ PlateauTracker + Catapult - Escape training plateaus")
print("  ✓ Catapult resets early stopping - Plateau epochs don't count against progression")
print("  ✓ compute_combined_loss_with_contrastive() - Model's built-in loss")
print("  ✓ Physics usage validation - Detect modality collapse")
print("  ✓ Sample display - See actual Q&A during training")
print(f"  ✓ Physics similarity threshold: {PHYSICS_SIM_WARNING_THRESHOLD}")
print("=" * 70)

In [ ]:

# ============================================================
# FIXED: SAMPLE DISPLAY - Uses MCQ scoring (not generation)
# ============================================================
def display_validation_samples_mcq(model, val_loader, device, num_samples=10, skip_descriptive=True):
    """
    Display MCQ evaluation using select_answer_from_choices (the method used in training).
    
    IMPORTANT: Model was trained with MCQ loss (scoring candidates), NOT text generation.
    So we MUST use select_answer_from_choices for evaluation, not forward().
    
    Args:
        model: The trained model
        val_loader: Validation DataLoader
        device: Device to run on
        num_samples: Number of samples to display
        skip_descriptive: If True, skip descriptive questions (count, exist, color, shape, material)
    """
    model.eval()
    
    # Descriptive question patterns to skip
    DESCRIPTIVE_PATTERNS = [
        "how many", "what color", "what shape", "what material",
        "what is the color", "what is the shape", "what is the material",
        "are there any", "is there a", "are there", "is there"
    ]
    
    def is_descriptive(question):
        q_lower = question.lower()
        return any(p in q_lower for p in DESCRIPTIVE_PATTERNS)
    
    print("=" * 70)
    print("MCQ EVALUATION (using select_answer_from_choices)")
    if skip_descriptive:
        print("[Excluding descriptive questions]")
    print("=" * 70)

    samples_shown = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in val_loader:
            if samples_shown >= num_samples:
                break

            physics_states = batch['physics_states'].to(device)
            object_mask = batch['object_mask'].to(device)
            questions = batch['questions']
            answers = batch['answers']
            choices_batch = batch.get('choices', [None] * len(questions))
            
            for i, (q, expected, choices) in enumerate(zip(questions, answers, choices_batch)):
                if samples_shown >= num_samples:
                    break
                
                # Skip descriptive questions if requested
                if skip_descriptive and is_descriptive(q):
                    continue
                
                # Skip if no choices available
                if choices is None or len(choices) == 0:
                    continue
                
                # Filter out [PAD] choices
                valid_choices = [c for c in choices if c != '[PAD]']
                if len(valid_choices) < 2:
                    continue
                
                # Get prediction using MCQ scoring
                states_single = physics_states[i:i+1]
                mask_single = object_mask[i:i+1]
                
                pred = model.select_answer_from_choices(
                    physics_states=states_single,
                    object_mask=mask_single,
                    question_text=[q],
                    choices=[valid_choices],
                    max_length=128
                )[0]
                
                # Check correctness
                pred_clean = pred.strip().lower()
                exp_clean = expected.strip().lower()
                is_correct = (pred_clean == exp_clean) or (exp_clean in pred_clean) or (pred_clean in exp_clean)
                
                status = "✓ CORRECT" if is_correct else "✗ WRONG"
                if is_correct:
                    correct += 1
                total += 1

                print(f"
[Sample {samples_shown + 1}] {status}")
                print(f"  Q: {q[:80]}{'...' if len(q) > 80 else ''}")
                print(f"  Choices:")
                for j, choice in enumerate(valid_choices):
                    marker = "→" if choice.lower().strip() == pred_clean else " "
                    correct_marker = "✓" if choice.lower().strip() == exp_clean else " "
                    print(f"    {marker}{correct_marker} [{j+1}] {choice}")
                print(f"  Expected: {expected}")
                print(f"  Predicted: {pred}")

                samples_shown += 1

    accuracy = 100.0 * correct / max(total, 1)
    print(f"
{'=' * 70}")
    print(f"MCQ Accuracy: {correct}/{total} ({accuracy:.1f}%)")
    print("=" * 70)
    return accuracy


# Replace the old function call with the new one
print("✅ Fixed evaluation function added: display_validation_samples_mcq()")
print("   - Uses MCQ scoring (select_answer_from_choices) NOT text generation")
print("   - Shows all choices with markers for predicted (→) and correct (✓)")
print("   - Excludes descriptive questions by default")



In [ ]:
# ============================================================
# CELL 14: PHASE 1 - TRAIN ADAPTER + HEADS
# ============================================================
print("="*70)
print("PHASE 1: Training Adapter + Heads (LLM frozen)")
print("="*70)

# Ensure LLM is frozen
adapter.set_training_phase('adapter')

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, adapter.parameters()),
    lr=PHASE1_LR,
    weight_decay=0.01
)

trainable = sum(p.numel() for p in adapter.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")
print(f"Learning rate: {PHASE1_LR}")

# Use train_phase_v2 with all technologies
train_phase_v2(
    adapter, train_loader, test_loader, optimizer, device,
    phase_name="Phase 1: Adapter + Numerical Head",
    max_epochs=20,
    patience=5,
    min_delta=0.001,
    checkpoint_dir=OUTPUT_DIR,
    phase_num=1,
    save_every_n_epochs=3,
    use_contrastive=USE_CONTRASTIVE,
    contrastive_weight=CONTRASTIVE_WEIGHT,
    validate_every_n_epochs=VALIDATE_PHYSICS_EVERY,
    physics_sim_threshold=PHYSICS_SIM_WARNING_THRESHOLD,
    use_amp=USE_AMP,
    scaler=scaler,
    use_plateau_tracker=True,
    show_samples_every_n_epochs=999  # Skip samples - LLM frozen
)

In [ ]:
# ============================================================
# CELL 15: PHASE 2 - UNFREEZE LLM OUTPUT LAYER
# ============================================================
print("="*70)
print("PHASE 2: Training + LLM Output Layer")
print("="*70)

# Unfreeze LLM output layer
adapter.set_training_phase('llm_head')

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, adapter.parameters()),
    lr=PHASE2_LR,
    weight_decay=0.01
)

trainable = sum(p.numel() for p in adapter.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")
print(f"Learning rate: {PHASE2_LR}")

train_phase_v2(
    adapter, train_loader, test_loader, optimizer, device,
    phase_name="Phase 2: LLM Head",
    max_epochs=15,
    patience=4,
    min_delta=0.0005,
    checkpoint_dir=OUTPUT_DIR,
    phase_num=2,
    save_every_n_epochs=3,
    use_contrastive=USE_CONTRASTIVE,
    contrastive_weight=CONTRASTIVE_WEIGHT,
    validate_every_n_epochs=VALIDATE_PHYSICS_EVERY,
    physics_sim_threshold=PHYSICS_SIM_WARNING_THRESHOLD,
    use_amp=USE_AMP,
    scaler=scaler,
    use_plateau_tracker=True,
    show_samples_every_n_epochs=999  # Skip samples - LLM partially frozen
)

In [ ]:
# ============================================================
# CELL 16: PHASE 3 - FULL LLM FINE-TUNING
# ============================================================
print("="*70)
print("PHASE 3: Full LLM Fine-tuning (DistilGPT-2)")
print("="*70)

# Unfreeze full LLM
adapter.set_training_phase('full')

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, adapter.parameters()),
    lr=PHASE3_LR,
    weight_decay=0.01
)

trainable = sum(p.numel() for p in adapter.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")
print(f"Learning rate: {PHASE3_LR}")

train_phase_v2(
    adapter, train_loader, test_loader, optimizer, device,
    phase_name="Phase 3: Full LLM (DistilGPT-2)",
    max_epochs=20,  # More epochs for full LLM fine-tuning
    patience=3,
    min_delta=0.0005,
    checkpoint_dir=OUTPUT_DIR,
    phase_num=3,
    save_every_n_epochs=3,
    use_contrastive=USE_CONTRASTIVE,
    contrastive_weight=CONTRASTIVE_WEIGHT,
    validate_every_n_epochs=VALIDATE_PHYSICS_EVERY,
    physics_sim_threshold=PHYSICS_SIM_WARNING_THRESHOLD,
    use_amp=USE_AMP,
    scaler=scaler,
    use_plateau_tracker=True,
    show_samples_every_n_epochs=3  # Show samples - LLM fully unfrozen
)

In [ ]:
# ============================================================
# CELL 17: FINAL EVALUATION
# ============================================================
print("="*70)
print("FINAL EVALUATION")
print("="*70)

accuracy = evaluate(adapter, test_loader, device, num_samples=500)
print(f"\nTest accuracy: {accuracy:.1f}%")

# Save final model
final_path = Path(OUTPUT_DIR) / "adapter_v2_final.pt"
torch.save({
    'model_state_dict': adapter.state_dict(),
    'physics_dim': hidden_dim,
    'num_prefix_tokens': NUM_PREFIX_TOKENS,
    'llm_name': 'distilgpt2',
    'accuracy': accuracy,
    'question_types': FOCUS_QUESTION_TYPES  # From config cell
}, final_path)

print(f"\nFinal model saved: {final_path}")
print(f"Total parameters: {sum(p.numel() for p in adapter.parameters()):,}")

In [ ]:
# ============================================================
# CELL 18: TEST INFERENCE
# ============================================================
print("="*70)
print("TEST INFERENCE")
print("="*70)

# Get a sample batch
adapter.eval()
sample_batch = next(iter(test_loader))

physics_states = sample_batch['physics_states'].to(device)
object_mask = sample_batch['object_mask'].to(device)
questions = sample_batch['questions']
answers = sample_batch['answers']

with torch.no_grad():
    predictions = adapter(physics_states, object_mask, questions)

print("\nSample predictions:")
print("-" * 70)
for i in range(min(5, len(questions))):
    q = questions[i]
    expected = answers[i]
    predicted = predictions[i]
    match = "✓" if predicted.lower().strip() == expected.lower().strip() else "✗"
    print(f"Q: {q[:60]}...")
    print(f"   Expected: {expected}")
    print(f"   Predicted: {predicted} {match}")
    print()